# Results Recommendation Audit

Purpose: review **every recommendation context** (not just settled BET rows) and surface weak pockets to attack.

Use this notebook when you want:
- a full recommendation ledger view,
- weak-point slices (side, rest, projected workload),
- actionable diagnostics before changing gates/calibration.

In [ ]:
from pathlib import Path
import sys
import polars as pl
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "production").exists():
    for parent in ROOT.parents:
        if (parent / "production").exists() and (parent / "src").exists():
            ROOT = parent
            break

sys.path.insert(0, str(ROOT / "src"))
from Python.notebook_analysis_utils import (
    has_over_clv_red_flag,
    keep_best_available_lines,
)

LEDGER_PATH = ROOT / "artifacts" / "odds_log" / "ledger.parquet"
PROJ_PATH = ROOT / "artifacts" / "projection_log" / "projections.parquet"


def show_table(df: pl.DataFrame, n: int = 200):
    display(df.head(n).to_pandas().round(3))


if not LEDGER_PATH.exists():
    raise FileNotFoundError(f"Missing {LEDGER_PATH}")

ledger = pl.read_parquet(LEDGER_PATH)
print(f"ledger rows={ledger.height} date_range={ledger['game_date'].min()} -> {ledger['game_date'].max()}")

if PROJ_PATH.exists():
    proj = (
        pl.read_parquet(PROJ_PATH)
        .with_columns(pl.col("game_date").cast(pl.Utf8).str.slice(0, 10))
        .select([c for c in [
            "game_date", "player_name", "days_rest", "projected_tbf", "expected_K",
            "is_out_of_support", "is_abbreviated_outing", "oos_reason"
        ] if c in pl.read_parquet(PROJ_PATH, n_rows=1).columns])
        .unique(subset=["game_date", "player_name"], keep="first")
    )
else:
    proj = pl.DataFrame()

print(f"projection rows={proj.height}")

ledger rows=594 date_range=2026-07-30 -> 2026-08-12
projection rows=413


In [2]:
base = ledger.with_columns(pl.col("game_date").cast(pl.Utf8).str.slice(0, 10))

if proj.height:
    base = base.join(proj, on=["game_date", "player_name"], how="left")

print("recommendation context mix")
summary = (
    base.group_by([c for c in ["snapshot", "status", "side", "passes_floor"] if c in base.columns])
    .agg(pl.len().alias("n"))
    .sort("n", descending=True)
)
show_table(summary, n=200)

cols = [
    c for c in [
        "game_date", "player_name", "book", "line", "side", "edge", "passes_floor", "status",
        "days_rest", "projected_tbf", "expected_K", "is_out_of_support", "oos_reason", "note"
    ] if c in base.columns
]
print("\nlatest recommendation rows (most recent 120)")
recent = base.sort([c for c in ["logged_at_utc", "game_date"] if c in base.columns], descending=True)
show_table(recent.select(cols), n=120)

recommendation context mix


,snapshot,status,side,passes_floor,n
0,bet,settled,under,False,167
1,bet,settled,over,False,151
2,bet,settled,under,True,131
3,bet,settled,over,True,91
4,bet,open,under,False,19
5,bet,open,over,False,11
6,bet,open,over,True,8
7,bet,open,under,True,6
8,bet,void,under,True,4
9,bet,void,over,True,4



latest recommendation rows (most recent 120)


,game_date,player_name,book,line,side,edge,passes_floor,status,days_rest,projected_tbf,expected_K,note
0,2026-08-12,Eric Lauer,fanduel,4.5,under,0.023213,False,open,7.0,23.743536,4.124419,
1,2026-08-12,George Klassen,fanduel,4.5,over,0.015454,False,open,5.0,20.221191,4.844070,
2,2026-08-12,Cal Quantrill,fanduel,3.5,over,0.005437,False,open,9.0,20.493927,3.863425,
3,2026-08-12,Eric Lauer,draftkings,4.5,under,0.046998,False,open,7.0,23.743536,4.124419,
4,2026-08-12,Cal Quantrill,draftkings,3.5,under,0.000839,False,open,9.0,20.493927,3.863425,
...,...,...,...,...,...,...,...,...,...,...,...,...
115,2026-08-10,Mike Soroka,fanduel,4.5,under,0.432162,False,settled,58.0,8.142749,1.686126,oos=projected_tbf<12 | close_unavailable=past_...
116,2026-08-10,Casey Mize,fanduel,4.5,over,0.104023,False,settled,5.0,23.035427,5.542001,close_unavailable=past_window_tip-178m
117,2026-08-10,Logan Henderson,fanduel,5.5,under,0.084398,False,settled,6.0,21.888562,4.838131,close_unavailable=past_window_tip-178m
118,2026-08-10,Jacob Lopez,fanduel,4.5,under,0.042800,False,settled,5.0,22.308219,4.341307,close_unavailable=past_window_tip-178m


In [3]:
settled_raw = base.filter((pl.col("status") == "settled") & (pl.col("stake").fill_null(0) > 0))
settled = keep_best_available_lines(settled_raw)
print(f"settled_raw_n={settled_raw.height} settled_best_line_n={settled.height}")
if settled.is_empty():
    print("No settled staked rows yet.")
else:
    settled = settled.with_columns([
        pl.when(pl.col("days_rest").is_null()).then(pl.lit("unknown"))
        .when(pl.col("days_rest") < 10).then(pl.lit("rest_<10"))
        .when(pl.col("days_rest") < 45).then(pl.lit("rest_10_44"))
        .otherwise(pl.lit("rest_45_plus")).alias("rest_bucket"),
        pl.when(pl.col("projected_tbf").is_null()).then(pl.lit("unknown"))
        .when(pl.col("projected_tbf") < 12).then(pl.lit("tbf_<12"))
        .when(pl.col("projected_tbf") < 15).then(pl.lit("tbf_12_15"))
        .otherwise(pl.lit("tbf_15_plus")).alias("tbf_bucket"),
    ])

    def slice_report(by: str) -> pl.DataFrame:
        return (
            settled.group_by(by)
            .agg(
                pl.len().alias("n"),
                pl.col("stake").sum().alias("stake"),
                pl.col("pnl").sum().alias("pnl"),
                (pl.col("pnl").sum() / pl.col("stake").sum()).alias("roi"),
                pl.col("clv_pp").drop_nulls().mean().alias("mean_clv_pp"),
            )
            .sort("n", descending=True)
        )

    print("weak-point scan by side")
    side_health = slice_report("side")
    show_table(side_health, n=20)
    if has_over_clv_red_flag(side_health):
        print("RED FLAG: over mean_clv_pp <= 0 (or missing).")

    print("\nweak-point scan by rest bucket")
    show_table(slice_report("rest_bucket"), n=20)

    print("\nweak-point scan by projected_tbf bucket")
    show_table(slice_report("tbf_bucket"), n=20)

weak-point scan by side


,side,n,stake,pnl,roi,mean_clv_pp
0,under,131,11041.883668,1780.334783,0.161235,0.003310
1,over,91,6280.393555,-128.287593,-0.020427,0.005037



weak-point scan by rest bucket


,rest_bucket,n,stake,pnl,roi,mean_clv_pp
0,rest_<10,203,15356.542692,2324.200224,0.151349,0.005690
1,rest_10_44,19,1965.734531,-672.153034,-0.341935,-0.010579



weak-point scan by projected_tbf bucket


,tbf_bucket,n,stake,pnl,roi,mean_clv_pp
0,tbf_15_plus,220,16975.682874,1422.384755,0.083790,0.005054
1,tbf_12_15,2,346.594349,229.662435,0.662626,-0.072347
